# S1.2 — 진단 run 정밀판 (W4/3/2): 역전 + proxy 예측

**원칙 (claude.ai 교정):** *신호 크기는 W3가, 정직함은 개편이 준다.* 개편은 결과를 **크게**가 아니라 **엄밀·정직하게** 만든다 — size 교란·순열검정은 W4 상관을 **줄일 수도** 있고, 그게 진실이면 그대로 받는다. (plateau 절단 수정만은 *측정오류 교정*이라 dtHd가 살아나면 그건 정당.)

**S1(pilot) → S1.2(정식 진단) 개편:**
- **★ size 교란 점검:** proxy(‖Hδ‖²·δᵀHδ 등)는 층 파라미터수 N에 비례(N항 합) → 회복도 큰 층일수록 클 수 있어 *상관이 곡률 때문인지 큰 층 때문인지* 모호. → **per-param 정규화 + N 통제 부분상관**으로 가른다 (partial이 raw보다 확 작으면 = size 탓).
- **loss-R + acc-R 동시** (proxy=loss곡률과 같은 공간).
- **고정 2500스텝 수렴앵커** (적응형 plateau 폐기 = 마라톤층 절단 제거; 스모크서 1200 미수렴 확인).
- **촘촘 t-grid {10,30,100,300,800,1500,2500}** + **여러 단기예산 역전**(inv_{10,30,100,300}→conv).
- **부트 CI + 순위재현**(자의적 SNR컷 대체; 순열 p는 보조) · **정규화 갭 breakdown** (FP−PTQ)/(FP−random)>0.8.
- **HVP 유한차분 교차검증**(실모델) · **FP32-same-budget 통제**(회복=양자화수정 vs 그냥 추가학습).

**규모:** W4/3/2 × 21층 × seed{0,1,2,3,4} × t 7점. 출력 `outputs/s1_2/`·`figures/s1_2/`. 엔진 코어 무재작성(검증됨) — 분석층만 수술 + W3. ~6–7h(steps 2500·seed 5; 스모크 ~33s/궤적).

In [ ]:
# --- Colab 셋업 ---
import os
REPO = '/content/26_Capstone'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/u-nsiq/26_Capstone.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO)
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- 엔진 + Drive + 경로 (S1.2 전용 출력) ---
from qat_engine import *
import numpy as np, matplotlib.pyplot as plt

try:
    from google.colab import drive; drive.mount('/content/drive')
    ART = '/content/drive/MyDrive/26_Capstone'
except Exception:
    ART = './_local_art'
for sub in ['checkpoints','outputs','figures','outputs/s1_2','figures/s1_2']:
    os.makedirs(f'{ART}/{sub}', exist_ok=True)
DATA_ROOT = f'{ART}/data'
CKPT      = f'{ART}/checkpoints/resnet18_cifar100_fp32.pt'
OUTDIR    = f'{ART}/outputs/s1_2'
FIG       = f'{ART}/figures/s1_2'
OUT       = f'{OUTDIR}/s1_2_runs.jsonl'
print('device', DEVICE, '| ART', ART, '| S1.2 out', OUTDIR)

In [ ]:
# --- S1.2 설정 ---
BITS          = [4, 3, 2]                       # 주력 / 빈무대 / 붕괴앵커 (W8 = S0 sanity로 격하)
SEEDS         = [0, 1, 2, 3, 4]                  # 5 seed: 부트 CI 의미있게(실질판정 ci_lo>noise_inv 받침). 느리면 [0,1,2]
GRID          = (10, 30, 100, 300, 800, 1500, 2500)  # 고정 t (스모크서 1200 미수렴 → 상향)
STEPS         = 2500                            # 고정 수렴앵커 (스모크서 1200 미수렴 → 2500)
SHORT, CONV   = 30, 2500                         # 주 비교점 (단기 vs 장기)
SHORT_LIST    = (10, 30, 100, 300)              # 역전을 여러 단기예산서 (Codex#3)
RANDOM_ACC    = 1.0                             # CIFAR-100 random → 정규화 갭 분모
LR            = 1e-3
N_BATCH_PROXY = 8                               # HVP/Fisher 추정 배치 (normHd2 편향↓)
print('bits', BITS, '| seeds', SEEDS, '| grid', GRID, '| short_list', SHORT_LIST)

## baseline + 층목록·비용(N) + λ_max(lr 타당성) + HVP 유한차분 교차검증

In [ ]:
train_loader, val_loader, calib_loader = get_loaders('cifar100', batch=128, calib_size=512, data_root=DATA_ROOT)
assert os.path.exists(CKPT), 'FP32 체크포인트 없음 — S0 먼저'
fp_model = load_model('resnet18', 'cifar100', ckpt=CKPT)
fp_acc = evaluate(fp_model, val_loader)
layers = list_quant_layers(make_ptq_model(fp_model, 4))
costs  = get_layer_costs(make_ptq_model(fp_model, 4), layers)   # N = 층 파라미터수 (size 통제용)
print(f'FP32 천장 {fp_acc:.2f} | 양자화 층 {len(layers)}개')

pm4 = make_ptq_model(fp_model, 4)
rep_layers = [layers[1], layers[len(layers)//2], layers[-1]]
print(f'--- lr 타당성: eta*|lambda_max| (eta={LR:.0e}), <1 이어야 (1-eta*lambda)^t 단조 ---')
for L in rep_layers:
    lam = lambda_max_layer(pm4, L, calib_loader, n_iter=12, n_batches=4)
    print(f'  {L:24s} lambda_max~{lam:+.3e}  eta*|lam|={LR*abs(lam):.3e}  {"OK(<1)" if LR*abs(lam)<1 else "WARN(>=1)"}')

# HVP 검증 = hvp_layer는 toy(additive-STE 위 float64 rel 3.8e-7) + S0(실모델 finite·non-zero·PSD부호) 인용.
# 유한차분 실모델 교차검증은 *제거*: fake-quant forward가 ±εδ를 재양자화 → FD가 dense H·δ 아닌 sparse 재양자화
# 점프를 재서 cos≪1 거짓경보(claude.ai). 정식 FD는 plain모델 weight=wq0+float64로 = 11월.
print('HVP 검증: toy 3.8e-7 + S0 실모델 인용 (위 lambda_max가 hvp_layer 실모델 동작 입증)')

In [ ]:
# δ / fake-quant 계산 검증 (Codex) — quant_error가 round/clamp/scale 공식대로 + round-err <= scale/2 인가.
# (proxy 전부가 quant_error에 의존 = foundation. '방향(FP로?)' 가정검증 아니라 '계산' 검증.)
deltas4 = quant_error(pm4)
print('δ 계산: max_delta_match~0(공식 일치), max_round_err_ratio<=0.5(clip 없으면)')
for L in rep_layers:
    m = dict(pm4.named_modules())[L]; qmax = 2 ** (m.n_bits - 1) - 1
    manual_q = torch.clamp(torch.round(m.weight / m.scale), -qmax, qmax) * m.scale
    match = (deltas4[L] - (m.weight - manual_q)).abs().max().item()
    rerr  = ((m.weight - manual_q).abs() / (m.scale + 1e-12)).max().item()
    print(f'  {L:24s} max_delta_match={match:.2e}  max_round_err_ratio={rerr:.3f}')

## 스모크 — 3층·1seed·W4 (loss-R / 고정 grid 경로 점검, ~몇 분)

In [ ]:
sm_layers = [layers[1], layers[len(layers)//2], layers[-1]]
pm = make_ptq_model(fp_model, 4)
ba, bl = evaluate_full(pm, val_loader)
print(f'W4 PTQ acc {ba:.2f} (gap {fp_acc-ba:.2f}%p) | loss {bl:.4f}')
px = proxy_scores(pm, fp_model, 4, calib_loader, layers=sm_layers, n_batches=N_BATCH_PROXY)
for L in sm_layers:
    m = make_ptq_model(fp_model, 4)
    set_trainable(m, [L])
    r = short_qat(m, train_loader, val_loader, steps=STEPS, eval_at=GRID, plateau=False,
                  seed=0, lr=LR, momentum=0.0, track_loss=True)
    racc  = {t: round(r[t]['acc'] - ba, 3) for t in r}
    rloss = {t: round(bl - r[t]['loss'], 4) for t in r}
    print(f'{L}\n   acc-R ={racc}\n   loss-R={rloss}  normHd2={px[L]["normHd2"]:.2e} dtHd={px[L]["dtHd"]:.2e}')
print('스모크 OK — full sweep로')

## 전체 sweep (W4/3/2 × 전층 × seed × 고정 grid) — Drive `outputs/s1_2/` 증분 저장. ~6–7h (steps 2500·seed 5; 스모크 ~33s/궤적).

In [ ]:
RESULTS = {}
import json as _json
for bit in BITS:
    pm = make_ptq_model(fp_model, bit)
    pa, pl = evaluate_full(pm, val_loader); gap = fp_acc - pa
    proxies = proxy_scores(pm, fp_model, bit, calib_loader, layers=layers, n_batches=N_BATCH_PROXY)
    _json.dump(proxies, open(f'{OUTDIR}/s1_2_proxies_w{bit}.json', 'w'))
    Racc = {L: {} for L in layers}; Rloss = {L: {} for L in layers}
    for L in layers:
        for s in SEEDS:
            m = make_ptq_model(fp_model, bit)        # PTQ 베이스라인 pa/pl 재사용(비트당 동일·결정적)
            set_trainable(m, [L])
            r = short_qat(m, train_loader, val_loader, steps=STEPS, eval_at=GRID, plateau=False,
                          seed=s, lr=LR, momentum=0.0, track_loss=True)
            for t in r:
                Racc[L].setdefault(t, []).append(r[t]['acc'] - pa)
                Rloss[L].setdefault(t, []).append(pl - r[t]['loss'])
            rec = {**{f'acc_{t}': r[t]['acc'] - pa for t in r}, **{f'loss_{t}': pl - r[t]['loss'] for t in r}}
            log_run({'phase': 'S1.2', 'bit': bit, 'layer': L, 'seed': s}, rec, path=OUT)
    RESULTS[bit] = dict(gap=gap, ptq_acc=pa, ptq_loss=pl, proxies=proxies, Racc=Racc, Rloss=Rloss)
    print(f'[W{bit}] gap {gap:.2f}%p, ptq_loss {pl:.3f} — {len(layers)}층 x {len(SEEDS)}seed 완료')
print('sweep 완료')

## 분석 + 그림 (loss-R 주, acc-R 보조)

In [ ]:
# 헬퍼: 회복행렬 · 신호검증 · 정규화갭 breakdown · per-param/부분상관(size) · 유의성
def _r(v):   return None if v is None or (isinstance(v,float) and np.isnan(v)) else round(float(v),3)
def _fmt(v): return ' nan ' if (v is None or np.isnan(v)) else f'{v:+.2f}'
def Rmat(bit, metric): return RESULTS[bit]['Racc'] if metric=='acc' else RESULTS[bit]['Rloss']
def mean_R(bit, t, metric='loss'):
    M = Rmat(bit, metric); return [float(np.mean(M[L][t])) for L in layers]
def std_R(bit, t, metric='loss'):
    M = Rmat(bit, metric); return [float(np.std(M[L][t])) for L in layers]
def per_seed(bit, t, metric='loss'):
    M = Rmat(bit, metric); return [[M[L][t][s] for L in layers] for s in range(len(SEEDS))]
def proxy_vec(bit, key): return [RESULTS[bit]['proxies'][L][key] for L in layers]
def proxy_pp(bit, key):  return [RESULTS[bit]['proxies'][L][key]/costs[L] for L in layers]  # per-parameter
def Nvec():              return [costs[L] for L in layers]

def signal_check(bit, t, metric='loss'):
    means = mean_R(bit, t, metric); noise = float(np.mean(std_R(bit, t, metric)))
    ps = per_seed(bit, t, metric)
    pairs = [spearman(ps[i], ps[j]) for i in range(len(SEEDS)) for j in range(i+1, len(SEEDS))]
    rs = float(np.nanmean(pairs)) if pairs else float('nan')
    return dict(spread=float(np.std(means)), seed_noise=noise, snr=float(np.std(means))/max(noise,1e-12), rank_stability=rs)

def normalized_gap(bit):                 # (FP - PTQ)/(FP - random): 회복가능 범위의 몇 % 무너졌나
    return (fp_acc - RESULTS[bit]['ptq_acc']) / (fp_acc - RANDOM_ACC)
def is_breakdown(bit):
    return normalized_gap(bit) > 0.8     # severe breakdown (Codex#5)

def inversion_report(bit, metric='loss'):
    sh = mean_R(bit, SHORT, metric); cv = mean_R(bit, CONV, metric)
    rel = perm_pvalue_related(sh, cv)   # 보조: short<->long 순위 관련성 (역전 유의성 p가 아님!)
    boot = bootstrap_inversion(per_seed(bit, SHORT, metric), per_seed(bit, CONV, metric))
    c30 = signal_check(bit, SHORT, metric); cpl = signal_check(bit, CONV, metric)
    noise_inv = 1.0 - min(c30['rank_stability'], cpl['rank_stability'])
    real = bool((not is_breakdown(bit)) and (c30['rank_stability'] > 0.5) and (cpl['rank_stability'] > 0.5)
                and (boot['ci_lo'] > noise_inv))   # 역전 판정 = 붕괴X + 순위재현 + inv CI가 noise 초과
    return dict(inv=boot['inversion'], ci=(boot['ci_lo'], boot['ci_hi']),
                rank_relation_rho=rel['rho'], rank_relation_p=rel['p_value'],
                noise_inv=noise_inv, breakdown=is_breakdown(bit), real_inversion=real,
                rs_short=c30['rank_stability'], rs_conv=cpl['rank_stability'])
print('헬퍼 OK (normalized_gap·proxy_pp·partial_spearman[엔진])')

In [ ]:
# FP32-same-budget 통제 (Codex#8) — 회복이 *양자화 수정*인가 그냥 *추가학습*인가
# W4 회복 top/mid/low 3층을 같은 스텝으로 FP32(76.84%)에서 학습 → 거의 안 오르면 회복=진짜 양자화수정
w4 = 4
mr_s = mean_R(w4, SHORT, 'acc'); mr_c = mean_R(w4, CONV, 'acc')
order = list(np.argsort(mr_c)[::-1]); idxs = [order[0], order[len(order)//2], order[-1]]
fp_ctrl = {}
print('  층                         QAT회복(conv)   FP32추가학습(conv)   판정')
for i in idxs:
    L = layers[i]
    m = clone_fp32_model(fp_model); set_trainable(m, [L])
    r = short_qat(m, train_loader, val_loader, steps=STEPS, eval_at=(SHORT, CONV), plateau=False,
                  seed=0, lr=LR, momentum=0.0, track_loss=True)
    fpc = r[CONV]['acc'] - fp_acc; fps = r[SHORT]['acc'] - fp_acc
    fp_ctrl[L] = dict(qat_short=mr_s[i], qat_conv=mr_c[i], fp_short=fps, fp_conv=fpc)
    verdict = '회복=양자화수정' if abs(fpc) < 0.5*abs(mr_c[i]) + 1e-9 else '추가학습 오염 의심'
    print(f'  {L:24s}  {mr_c[i]:+8.2f}%p      {fpc:+8.2f}%p        {verdict}')
import json as _json
_json.dump(fp_ctrl, open(f'{OUTDIR}/s1_2_fp32_control.json', 'w'))
print('→ FP32 변화 << QAT 회복이면 회복은 추가학습이 아니라 양자화오차 수정 (연구계획서 §5 통제)')

In [ ]:
# FIG0 — 신호검증 (loss-R): 층별 회복 mean±std(seed) (기술통계, 참고)
chk = [b for b in BITS if RESULTS[b]['gap'] > 0.5]
fig, axs = plt.subplots(len(chk), 2, figsize=(12, 4*len(chk)), squeeze=False)
for r_, bit in enumerate(chk):
    for c_, t in enumerate([SHORT, CONV]):
        m = np.array(mean_R(bit, t, 'loss')); sd = np.array(std_R(bit, t, 'loss')); o = np.argsort(m)[::-1]
        ck = signal_check(bit, t, 'loss')
        ax = axs[r_][c_]; ax.errorbar(range(len(layers)), m[o], yerr=sd[o], fmt='o', ms=3, capsize=2)
        ax.set_xlabel('layer (sorted)'); ax.set_ylabel('loss-recovery')
        ax.set_title(f'W{bit} t={t}  SNR={ck["snr"]:.1f} rank-stab={ck["rank_stability"]:.2f}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_signal_check.png', dpi=120); plt.show()
print('SNR/rank-stab = 기술통계. 역전 판정 = 순위재현 + 부트 CI(ci_lo>noise_inv); 순열 p는 보조.')

In [ ]:
# 전 grid SNR/rank-stab 라벨 표 (Codex#2) — t=30 하나로 안 박고 여러 단기예산 안정성 함께
LABEL = {10:'ultra-short', 30:'practical', 100:'mod-short', 300:'long-short', 800:'near-conv', 1500:'long', 2500:'converged'}
for bit in [b for b in BITS if RESULTS[b]['gap'] > 0.5]:
    print(f'W{bit} (loss-R) 단기예산별 신호:')
    for t in GRID:
        ck = signal_check(bit, t, 'loss')
        print(f'   t={t:4d} [{LABEL.get(t,""):11s}] SNR={ck["snr"]:5.1f}  rank-stab={ck["rank_stability"]:+.2f}')

In [ ]:
# 임계값 민감도 — 결론(breakdown·rank_stab 통과)이 임의 컷(0.5/0.8 등)에 흔들리나 (Codex#7)
print('임계값 민감도 (결론이 임계값에 robust한가):')
for bit in BITS:
    ng = normalized_gap(bit)
    c30 = signal_check(bit, SHORT, 'loss'); cpl = signal_check(bit, CONV, 'loss')
    bd = ['%.1f:%s' % (th, ng > th) for th in (0.7, 0.8, 0.9)]
    rs = ['%.1f:%s' % (th, (c30['rank_stability'] > th and cpl['rank_stability'] > th)) for th in (0.4, 0.5, 0.6)]
    print(f" W{bit}: breakdown[{', '.join(bd)}]  rank_stab_pass[{', '.join(rs)}]")
print('→ 같은 비트서 임계값 바꿔도 True/False 안 바뀌면 결론이 임계값에 안 의존(robust).')

In [ ]:
# 수렴 평탄성 + 단조성 — |loss-R(conv)-loss-R(직전)| 작고 conv가 max여야 수렴앵커 타당
t1, t2 = GRID[-2], GRID[-1]
fig, ax = plt.subplots(figsize=(7,4))
for bit in [b for b in BITS if RESULTS[b]['gap'] > 0.5]:
    a1 = np.array(mean_R(bit, t1, 'loss')); a2 = np.array(mean_R(bit, t2, 'loss'))
    d = np.abs(a2 - a1); n_drop = int(np.sum(a2 < a1 - 1e-6))
    ax.plot(sorted(d, reverse=True), marker='o', ms=3, label=f'W{bit} (maxΔ={d.max():.3f}, 비단조 {n_drop}층)')
ax.set_xlabel('layer (sorted)'); ax.set_ylabel(f'|loss-R({t2})-loss-R({t1})|'); ax.legend()
ax.set_title(f'convergence flatness ({t1}->{t2}; 작을수록 수렴 타당)')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_flatness.png', dpi=120); plt.show()
print(f'maxΔ 크거나 비단조 층 많으면 {t2}서도 수렴 전 — 해석 주의.')

In [ ]:
# 역전 (RQ2) — loss-R, 부트 CI + 순위재현(순열 p는 보조 rank_relation_p). rank 1=best
from scipy.stats import rankdata
def best_rank(v): return len(v) + 1 - rankdata(v)
inv_bits = [b for b in BITS if RESULTS[b]['gap'] > 0.5]
fig, axs = plt.subplots(1, len(inv_bits), figsize=(5*len(inv_bits), 4.8), squeeze=False)
for c_, bit in enumerate(inv_bits):
    rep = inversion_report(bit, 'loss')
    sh = mean_R(bit, SHORT, 'loss'); cv = mean_R(bit, CONV, 'loss')
    ax = axs[0][c_]; ax.scatter(best_rank(sh), best_rank(cv), s=25)
    lim = [0, len(layers)+1]; ax.plot(lim, lim, '--', color='gray', lw=1)
    ax.set_xlabel(f'short (t={SHORT}) rank'); ax.set_ylabel(f'converged (t={CONV}) rank')
    tag = 'BREAKDOWN' if rep['breakdown'] else ('REAL inv' if rep['real_inversion'] else 'n.s.')
    ax.set_title(f'W{bit} inv={rep["inv"]:.2f} vs noise={rep["noise_inv"]:.2f} CI[{rep["ci"][0]:.2f},{rep["ci"][1]:.2f}]\nrank-rel p={rep["rank_relation_p"]:.3f}  {tag}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_inversion.png', dpi=120); plt.show()

In [ ]:
# 여러 단기예산서 역전 (Codex#3): 단기 t -> conv 각각 inv + 순열 p ("t=30은 noisy해도 t=100부터 안정" 같은 해석)
print(f'단기예산별 역전 (vs conv={CONV}):')
for bit in inv_bits:
    print(f' W{bit} (breakdown={is_breakdown(bit)}, norm_gap={normalized_gap(bit):.2f}):')
    cv = mean_R(bit, CONV, 'loss')
    for ts in SHORT_LIST:
        pr = perm_pvalue_related(mean_R(bit, ts, 'loss'), cv)
        inv = (1 - pr['rho']) if not np.isnan(pr['rho']) else float('nan')
        print(f'   t={ts:4d}->conv  inv={inv:.2f}  rank_rel_p={pr["p_value"]:.3f}')

In [ ]:
# proxy↔회복 산점도 (loss-R, 주력 W4)
bit = 4
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
sh = mean_R(bit, SHORT, 'loss'); nh = proxy_vec(bit, 'normHd2')
axs[0].scatter(nh, sh, s=25); axs[0].set_xscale('log')
axs[0].set_xlabel('||H d||^2 (short)'); axs[0].set_ylabel(f'loss-R({SHORT})'); axs[0].set_title(f'short proxy  rho={spearman(nh,sh):.2f}')
cv = mean_R(bit, CONV, 'loss'); dh = proxy_vec(bit, 'dtHd')
axs[1].scatter(dh, cv, s=25)
axs[1].set_xlabel('dT H d (converged)'); axs[1].set_ylabel(f'loss-R({CONV})'); axs[1].set_title(f'converged proxy  rho={spearman(dh,cv):.2f}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_proxy_scatter_w{bit}.png', dpi=120); plt.show()

In [ ]:
# rho-vs-t 교차 (RQ4 핵심): normHd2(단기) vs dtHd(수렴) — loss-R, raw
fig, axs = plt.subplots(1, len(inv_bits), figsize=(5*len(inv_bits), 4.2), squeeze=False)
for c_, bit in enumerate(inv_bits):
    nh = proxy_vec(bit, 'normHd2'); dh = proxy_vec(bit, 'dtHd')
    rs = [spearman(nh, mean_R(bit, t, 'loss')) for t in GRID]
    rc = [spearman(dh, mean_R(bit, t, 'loss')) for t in GRID]
    ax = axs[0][c_]; ax.plot(GRID, rs, marker='o', label='||Hd||^2 (short)'); ax.plot(GRID, rc, marker='s', label='dT H d (conv)')
    ax.set_xscale('log'); ax.set_xlabel('t'); ax.set_ylabel('rho(proxy, loss-R)'); ax.set_title(f'W{bit}: which proxy wins when'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_rho_vs_t.png', dpi=120); plt.show()
print('교차(normHd2 단기우세 -> dtHd 수렴우세)면 lambda^2<->lambda 직접 증거.')

In [ ]:
# RQ4 — proxy↔회복: raw / per-param / N-통제 부분상관 (★ size 교란 점검). loss-R
PROX = ['normHd2', 'dtHd', 'fisher', 'werr', 'iso_out']
import csv as _csv
N = Nvec()
for bit in inv_bits:
    print(f'=== W{bit} (loss-R)  raw | per-param | partial(|N)   @ short={SHORT}, conv={CONV} ===')
    rows = []
    for k in PROX:
        raw = proxy_vec(bit, k); pp = proxy_pp(bit, k)
        ys = mean_R(bit, SHORT, 'loss'); yc = mean_R(bit, CONV, 'loss')
        print(f'  {k:9s} short {_fmt(spearman(raw,ys))}/{_fmt(spearman(pp,ys))}/{_fmt(partial_spearman(raw,ys,N))}   conv {_fmt(spearman(raw,yc))}/{_fmt(spearman(pp,yc))}/{_fmt(partial_spearman(raw,yc,N))}')
        for t in GRID:
            y = mean_R(bit, t, 'loss')
            rows.append([k, t, _r(spearman(raw, y)), _r(spearman(pp, y)), _r(partial_spearman(raw, y, N))])
    with open(f'{OUTDIR}/s1_2_proxy_rho_w{bit}.csv', 'w', newline='') as f:
        w = _csv.writer(f); w.writerow(['proxy','t','rho_raw','rho_perparam','rho_partial_N']); w.writerows(rows)
print('★ partial(|N) << raw 이면 상관이 곡률 아니라 층크기(N) 탓. nan=proxy가 N과 거의 동일(=완전 size).')

In [ ]:
# δ/이론 검증 (계획서 §6, 올바른 버전) — 이론 예측 회복 ½·δᵀHδ vs 실측 loss-R(수렴).
# 제거한 기하버전과 달리 tautology 아님: HVP가 예측한 회복 *크기*가 실제와 맞나 직접 비교(calibration).
fig, axs = plt.subplots(1, len(inv_bits), figsize=(5*len(inv_bits), 4.2), squeeze=False)
for c_, bit in enumerate(inv_bits):
    pred = [0.5 * RESULTS[bit]['proxies'][L]['dtHd'] for L in layers]   # ½ δᵀHδ = 2차이론 예측 회복(loss)
    meas = mean_R(bit, CONV, 'loss')                                    # 실측 수렴 회복(loss)
    ratio = [m / p for m, p in zip(meas, pred) if p > 1e-12]
    med = float(np.median(ratio)) if ratio else float('nan')
    ax = axs[0][c_]; ax.scatter(pred, meas, s=25)
    lo = min(pred + meas); hi = max(pred + meas); ax.plot([lo, hi], [lo, hi], '--', color='gray', lw=1)
    ax.set_xlabel('predicted  0.5*dT H d'); ax.set_ylabel(f'measured loss-R({CONV})')
    ax.set_title(f'W{bit}: theory calib  rho={spearman(pred, meas):.2f}  meas/pred~{med:.0f}x')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_theory_calib.png', dpi=120); plt.show()
print('y=x 근처=δ+2차이론이 회복 *크기*까지 예측. 실측≫예측=회복이 de-quant 이상(보상 최적화). W2서 깨지면 비트의존.')

In [ ]:
# RQ6 — iso_out vs 회복 (loss-R, 수렴). 음수=출력변화 작은 층이 더 회복
bit = 4; io = proxy_vec(bit, 'iso_out'); rp = mean_R(bit, CONV, 'loss'); rho = spearman(io, rp)
plt.figure(figsize=(6.5, 4.5)); plt.scatter(io, rp, s=25); plt.xscale('log')
plt.xlabel('isolated ||d logit||^2'); plt.ylabel(f'loss-R({CONV})'); plt.title(f'W{bit} RQ6 rho={rho:.2f}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_rq6_w{bit}.png', dpi=120); plt.show()
print(f'RQ6 rho={rho:.3f} ->', 'NEG(재현)' if rho < 0 else 'POS/0(반박·무관)')

In [ ]:
# 갭·역전 vs 비트 (W4/3/2 + 기존 W8은 인용)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(BITS, [RESULTS[b]['gap'] for b in BITS], marker='o'); axs[0].set_xlabel('bit'); axs[0].set_ylabel('PTQ gap'); axs[0].set_title('gap vs bit'); axs[0].invert_xaxis()
for bit in inv_bits:
    rep = inversion_report(bit, 'loss'); col = 'red' if rep['breakdown'] else ('C0' if rep['real_inversion'] else 'orange')
    yerr = [[max(rep['inv']-rep['ci'][0], 0)], [max(rep['ci'][1]-rep['inv'], 0)]]
    axs[1].errorbar([bit], [rep['inv']], yerr=yerr, fmt='o', color=col, capsize=4)
axs[1].set_xlabel('bit'); axs[1].set_ylabel('inversion (loss-R) +/-95%CI'); axs[1].set_title('inversion vs bit (blue=real, orange=n.s., red=breakdown)'); axs[1].invert_xaxis()
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_vs_bit.png', dpi=120); plt.show()

In [ ]:
# 요약 저장 — s1_2_summary.json (loss-R, 순열p·부트CI·partial·multi-inv·norm_gap) + recovery csv
import json as _json, csv as _csv
def _safe(v): return None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), 4)
N = Nvec()
summary = {'fp_acc': fp_acc,
           'config': dict(bits=BITS, seeds=SEEDS, grid=list(GRID), steps=STEPS, short=SHORT, conv=CONV,
                          short_list=list(SHORT_LIST), n_batch_proxy=N_BATCH_PROXY),
           'bits': {}}
for bit in BITS:
    rep = inversion_report(bit, 'loss') if RESULTS[bit]['gap'] > 0.5 else None
    c30 = signal_check(bit, SHORT, 'loss'); cpl = signal_check(bit, CONV, 'loss')
    cv = mean_R(bit, CONV, 'loss') if RESULTS[bit]['gap'] > 0.5 else None
    inv_by_short = {str(ts): _safe(1 - perm_pvalue_related(mean_R(bit, ts, 'loss'), cv)['rho']) for ts in SHORT_LIST} if cv else None
    _cpred = [0.5 * RESULTS[bit]['proxies'][L]['dtHd'] for L in layers]; _cmeas = mean_R(bit, CONV, 'loss')
    _cr = [m / p for m, p in zip(_cmeas, _cpred) if p > 1e-12]
    summary['bits'][str(bit)] = dict(
        gap=round(RESULTS[bit]['gap'], 3), ptq_acc=round(RESULTS[bit]['ptq_acc'], 2), ptq_loss=round(RESULTS[bit]['ptq_loss'], 4),
        normalized_gap=_safe(normalized_gap(bit)), breakdown=bool(is_breakdown(bit)),
        snr_short=_safe(c30['snr']), snr_conv=_safe(cpl['snr']),
        rank_stab_short=_safe(c30['rank_stability']), rank_stab_conv=_safe(cpl['rank_stability']),
        inversion=_safe(rep['inv']) if rep else None,
        inv_ci_lo=_safe(rep['ci'][0]) if rep else None, inv_ci_hi=_safe(rep['ci'][1]) if rep else None,
        rank_relation_p=_safe(rep['rank_relation_p']) if rep else None, noise_inv=_safe(rep['noise_inv']) if rep else None,
        real_inversion=bool(rep['real_inversion']) if rep else None,
        inv_by_short=inv_by_short,
        rho_short_normHd2_raw=_safe(spearman(proxy_vec(bit,'normHd2'), mean_R(bit,SHORT,'loss'))),
        rho_short_normHd2_partialN=_safe(partial_spearman(proxy_vec(bit,'normHd2'), mean_R(bit,SHORT,'loss'), N)),
        rho_conv_dtHd_raw=_safe(spearman(proxy_vec(bit,'dtHd'), mean_R(bit,CONV,'loss'))),
        rho_conv_dtHd_partialN=_safe(partial_spearman(proxy_vec(bit,'dtHd'), mean_R(bit,CONV,'loss'), N)),
        theory_calib_rho=_safe(spearman(_cpred, _cmeas)),
        theory_calib_median_ratio=_safe(np.median(_cr)) if _cr else None)
    for metric in ['loss', 'acc']:
        with open(f'{OUTDIR}/s1_2_recovery_{metric}_w{bit}.csv', 'w', newline='') as f:
            w = _csv.writer(f); w.writerow(['layer','N'] + [f't{t}' for t in GRID] + [f'std{t}' for t in GRID])
            for L in layers:
                w.writerow([L, costs[L]] + [round(np.mean(Rmat(bit, metric)[L][t]), 4) for t in GRID] + [round(np.std(Rmat(bit, metric)[L][t]), 4) for t in GRID])
_json.dump(summary, open(f'{OUTDIR}/s1_2_summary.json', 'w'), indent=2, ensure_ascii=False)
print('저장 s1_2_summary.json + recovery csv (N 포함)')
for b in summary['bits']:
    s = summary['bits'][b]
    print(f"W{b} | norm_gap {s['normalized_gap']} | inv {s['inversion']} (noise {s['noise_inv']}) CI[{s['inv_ci_lo']},{s['inv_ci_hi']}] rank_rel_p {s['rank_relation_p']} REAL {s['real_inversion']} | normHd2 raw/partN {s['rho_short_normHd2_raw']}/{s['rho_short_normHd2_partialN']}")

In [ ]:
# 비교 — 기존 S1(W8/W4/W2, acc·적응형) vs S1.2(W4/3/2, loss·고정·순열p·partial). 읽기전용
import json as _json
prev = f'{ART}/outputs/s1_summary.json'; new = f'{OUTDIR}/s1_2_summary.json'
print('[기존 S1 — acc-R, 적응형 plateau, SNR컷]')
if os.path.exists(prev):
    for b, s in sorted(_json.load(open(prev))['bits'].items(), key=lambda x: -int(x[0])):
        print(f"  W{b} gap {s['gap']} inv {s.get('inversion_strength')} TRUST {s.get('inversion_trustworthy')} breakdown {s.get('breakdown')}")
else:
    print('  (s1_summary.json 없음)')
print('[S1.2 — loss-R, 고정2500, 순열p+부트CI, size-통제]')
for b, s in sorted(_json.load(open(new))['bits'].items(), key=lambda x: -int(x[0])):
    print(f"  W{b} norm_gap {s['normalized_gap']} inv {s['inversion']} (noise {s['noise_inv']}) rank_rel_p {s['rank_relation_p']} REAL {s['real_inversion']} | normHd2 raw->partN {s['rho_short_normHd2_raw']}->{s['rho_short_normHd2_partialN']}")

## S1.2 완료 — 무엇을 볼까 (★ size 교란부터)

1. **★ proxy가 곡률인가 size인가 (RQ4, `s1_2_proxy_rho_w*.csv`):** `rho_partial_N`(N 통제)이 `rho_raw`보다 **확 작으면** 상관은 *곡률* 아니라 *큰 층* 탓 — 솔직히 받는다. `nan`이면 proxy가 N과 거의 동일(완전 size). partial이 살아남아야 "곡률이 예측한다" 주장 가능.
2. **dtHd가 살아났나 (`s1_2_rho_vs_t`):** dtHd가 수렴쪽서 normHd2 추월하면 → S1의 dtHd 약함이 *plateau 절단 탓*이었음 확정 + λ²↔λ 교차(척추). (단 size 통제 후에도 성립해야 진짜.)
3. **역전이 REAL인가 (`s1_2_summary`):** `real_inversion`=(붕괴 아님)∧(순위재현 rank-stab>0.5)∧(부트 CI 하한>noise_inv). 순열 p(=rank_relation_p)는 보조(short↔long 관련성)지 역전 p 아님 —. `inv_by_short`로 "t=30 noisy해도 t=100부터 안정" 같이 판단.
4. **회복=양자화수정인가 (`s1_2_fp32_control.json`):** FP32 추가학습 변화 << QAT 회복이어야 (추가학습 오염 아님).
5. **HVP:** cell5 lambda_max로 실모델 hvp_layer 동작 확인(toy 3.8e-7+S0 인용). **수렴 타당:** flatness(1500→2500) 평탄·단조여야.
6. **W2:** norm_gap>0.8이면 breakdown 앵커.

**임계값 민감도표:** 결론(breakdown·rank_stab)이 0.5/0.8 컷에 안 흔들리나. (§6 δ는 3축 검증[Codex]: [계산] quant_error가 W-Q(W) 공식·round-err<=scale/2 맞나=δ단위sanity 셀(baseline 뒤) · [방향=FP로?] Q(W)가 W 최근접 grid라 tautology→제거 · [proxy 유효] 이론예측 ½δᵀHδ vs 실측 회복 calibration 셀, y=x 근처면 크기까지 예측. 정밀 기하버전=LSQ 11월.)

**해석 주의 (claude.ai):** ① W4 `real_inversion`이 False여도 *실패 아니라 marginality* (inv≈0.30이 노이즈바닥 noise_inv≈0.28에 아슬 — loss-R이 더 매끄러우면 통과할 수도). ② boolean 맹신 말고 **inv vs noise_inv 직접 + inv_by_short**(t=100/300이 t=30보다 깨끗한가) 읽기 — 부트 CI는 이제 seed 5개라 더 받침. ③ **진짜 베팅은 W3** (갭 크면 역전이 노이즈에서 확실히 떨어져 나옴).

**원칙 재확인:** 신호는 W3가, 정직함은 개편이 준다. 일부 상관이 줄어도(특히 size 통제 후) 그게 진실 — 발표에선 "size까지 통제하고도 곡률이 예측"이라 말할 수 있으면 훨씬 단단. → 결과 갖고 와서 해석.